# Deliverables 6, 10 and 11 — Classical comparator, hardware-specific FT scenarios, and crossover

Measures the independent spectral comparator with warmups/repetitions and combines measured classical costs with labeled FT proxies. This is a no-go/crossover sensitivity experiment, not a quantum-advantage claim.

In [1]:
from pathlib import Path
import sys
repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
import json, platform, numpy as np, pandas as pd
from quantum_aero.classical import LBMConfig, run_lbm
from quantum_aero.deliverables import run_pseudospectral_tgv

classical=[]
for re in (10,100,400,1000,2000,5000):
 for n in (32,64):
  cfg=LBMConfig(n=n,reynolds=re,t_end=1,mach=.05,snapshots=2)
  run_pseudospectral_tgv(cfg,repeats=1); run_lbm(cfg) # warmups
  sp=run_pseudospectral_tgv(cfg,repeats=5)
  lb=[run_lbm(cfg) for _ in range(5)]; last=lb[-1]["records"][-1]
  classical += [
   {"solver":"spectral","reynolds":re,"n":n,"runtime_median_seconds":sp["runtime_median_seconds"],"runtime_min_seconds":sp["runtime_min_seconds"],"runtime_max_seconds":sp["runtime_max_seconds"],"relative_l2":sp["relative_l2"],"memory_bytes":sp["memory_bytes"]},
   {"solver":"LBM","reynolds":re,"n":n,"runtime_median_seconds":float(np.median([x["runtime_seconds"] for x in lb])),"runtime_min_seconds":min(x["runtime_seconds"] for x in lb),"runtime_max_seconds":max(x["runtime_seconds"] for x in lb),"relative_l2":last["relative_l2"],"memory_bytes":lb[-1]["population_memory_bytes"]}]
classical_df=pd.DataFrame(classical); classical_df.to_csv(output_dir/"11_classical_comparator_t1.csv",index=False)
classical_df

,solver,reynolds,n,runtime_median_seconds,runtime_min_seconds,runtime_max_seconds,relative_l2,memory_bytes
0,spectral,10,32,0.184305,0.179725,0.195443,1.636399e-09,98304
1,LBM,10,32,0.775792,0.751652,0.796873,4.634948e-03,73728
2,spectral,10,64,0.649806,0.622883,1.338119,1.020680e-10,393216
3,LBM,10,64,4.312776,4.261615,4.641400,3.276732e-03,294912
4,spectral,100,32,0.198077,0.184116,0.391189,1.682294e-09,98304
5,LBM,100,32,0.812692,0.799283,0.859196,5.710993e-03,73728
6,spectral,100,64,0.651474,0.633028,0.685711,1.051223e-10,393216
7,LBM,100,64,4.800934,4.318865,5.300987,4.431667e-03,294912
8,spectral,400,32,0.179817,0.173961,0.187941,1.697233e-09,98304
9,LBM,400,32,0.784120,0.745474,0.822235,5.643492e-03,73728


In [3]:
factor=json.loads((output_dir/"06_structured_collision_ft_estimates.json").read_text())
post=pd.read_csv(output_dir/"05_postselection_summary.csv")
rows=[]
for scenario in factor["scenarios"]:
 for re in (10,100,400,1000,2000,5000):
  # Use nearest measured post-selection row; 2000/5000 conservatively use Re=1000.
  p=float(post.iloc[(post.reynolds-re).abs().argsort()[:1]].p_min.iloc[0])
  aa=np.ceil(np.pi/(4*np.sqrt(p)))
  classical_best=classical_df[(classical_df.reynolds==re)].runtime_median_seconds.min()
  for nt in (10,100,1000):
   tq=nt*scenario["block_query_time_seconds_proxy"]*(2*aa+1)
   rows.append({"scenario":scenario["name"],"reynolds":re,"timesteps":nt,"p":p,"aa_iterations_proxy":aa,
                "quantum_collision_only_seconds_proxy":tq,"best_measured_classical_seconds":classical_best,
                "ratio_quantum_collision_only_to_classical":tq/classical_best,
                "omitted_quantum_costs":"state preparation, global streaming, extraction, routing between blocks"})
crossover_df=pd.DataFrame(rows); crossover_df.to_csv(output_dir/"11_crossover_no_go.csv",index=False)
crossover_df.groupby(["scenario","timesteps"]).ratio_quantum_collision_only_to_classical.agg(["min","median","max"])

min        median           max
scenario    timesteps                                         
base        10           12.230377     13.111115     13.472352
            100         122.303773    131.111147    134.723518
            1000       1223.037728   1311.111473   1347.235181
optimistic  10            0.611519      0.655556      0.673618
            100           6.115189      6.555557      6.736176
            1000         61.151886     65.555574     67.361759
pessimistic 10           97.843018    104.888918    107.778814
            100         978.430183   1048.889178   1077.788145
            1000       9784.301826  10488.891780  10777.881445

In [4]:
meta={"hardware":platform.platform(),"processor":platform.processor(),"python":platform.python_version(),
"gpu_status":"not measured; no project GPU implementation is present",
"fv_status":"not measured; independent Fourier-vorticity RK4 is the implemented high-order comparator",
"ft_warning":"collision-only lower-bound proxy; absence of global streaming/preparation/extraction makes any favorable ratio insufficient for advantage"}
(output_dir/"11_crossover_metadata.json").write_text(json.dumps(meta,indent=2))
assert len(classical_df)==24 and len(crossover_df)==54
print("PASS: measured classical comparator and FT sensitivity/no-go ledger completed.")

PASS: measured classical comparator and FT sensitivity/no-go ledger completed.
